# Template
> python src/preprocess.py -input demo/Cancer_Data.tsv -out demo/Cancer_Data_processed.tsv

> python src/embedding_lookup_table.py -input demo/Cancer_Data_processed.tsv -out demo/Cancer_Data_embedding.npz

> python src/predict.py -input demo/Cancer_Data_processed.tsv -id demo/Cancer_Data_ID.tsv -input_embed demo/Cancer_Data_embedding.npz -train_embed data/disease_desc_embedding.npz -model bins/MONDO_0000167__model.pkl -out results/

> cat results/Cancer_Data_preds.csv

# Ready to use

In [1]:
import os, glob, subprocess, datetime
from typing import List, Sequence, Optional
import pandas as pd
from tqdm import tqdm

def run_species_metadata_pipeline(
    specie: str,
    metadata: Sequence[str],
    *,
    demo_dir: str = "demo",
    src_dir: str = "src",
    bins_dir: str = "bins",
    data_dir: str = "data",
    results_dir: str = "results",
    overwrite: bool = False,
) -> str:
    """
    Runs:
      1) preprocess.py
      2) embedding_lookup_table.py
      3) predict.py for each bins/MONDO_*__model.pkl
      4) merge per-model csv into one wide csv

    Returns:
      path to merged preds csv
    """
    specie = specie.strip()
    meta = [m.strip() for m in metadata if str(m).strip()]
    if not specie:
        raise ValueError("specie is empty")
    if not meta:
        raise ValueError("metadata is empty")

    meta_tag = "_".join(meta)  # e.g. "title_summary"
    # Inputs
    input_tsv = os.path.join(demo_dir, f"{specie}_{meta_tag}.tsv")
    id_tsv = os.path.join(demo_dir, f"{specie}_ID.tsv")

    # Intermediates
    processed_tsv = os.path.join(demo_dir, f"{specie}_{meta_tag}_processed.tsv")
    input_embed = os.path.join(demo_dir, f"{specie}_{meta_tag}_embedding.npz")

    # Shared resources
    train_embed = os.path.join(data_dir, "disease_desc_embedding.npz")
    models = sorted(glob.glob(os.path.join(bins_dir, "MONDO_*__model.pkl")))

    # Outputs
    out_dir = os.path.join(results_dir, f"{specie}_{meta_tag}")
    log_path = os.path.join(out_dir, "predict_all.log")
    merged_out = os.path.join(results_dir, f"{specie}_{meta_tag}_preds.csv")

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # 1) preprocess
    if overwrite or (not os.path.exists(processed_tsv)):
        subprocess.run(
            ["python", os.path.join(src_dir, "preprocess.py"),
             "-input", input_tsv,
             "-out", processed_tsv],
            check=True
        )

    # 2) embedding lookup table
    if overwrite or (not os.path.exists(input_embed)):
        subprocess.run(
            ["python", os.path.join(src_dir, "embedding_lookup_table.py"),
             "-input", processed_tsv,
             "-out", input_embed],
            check=True
        )

    # 3) predict for each model
    for model in tqdm(models, desc="Models"):
        mondo = os.path.basename(model).split("__model.pkl")[0]

        # skip if already produced (same behavior as your original)
        done = glob.glob(os.path.join(out_dir, f"{mondo}*.csv"))
        if done and (not overwrite):
            continue

        with open(log_path, "a", encoding="utf-8") as log:
            ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"\n[{ts}] START {mondo}\n")
            log.flush()

            try:
                subprocess.run(
                    ["python", os.path.join(src_dir, "predict.py"),
                     "-input", processed_tsv,
                     "-id", id_tsv,
                     "-input_embed", input_embed,
                     "-train_embed", train_embed,
                     "-model", model,
                     "-out", out_dir + "/"],
                    stdout=log, stderr=log, check=True
                )
                ts2 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                log.write(f"[{ts2}] DONE  {mondo}\n")
                log.flush()

            except subprocess.CalledProcessError as e:
                ts3 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                log.write(f"[{ts3}] FAIL  {mondo} (exit={e.returncode})\n")
                log.flush()
                raise

    # 4) stack all csv in out_dir -> merged_out (in results/)
    files = sorted(glob.glob(os.path.join(out_dir, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No per-model CSV found in {out_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        model_name = os.path.basename(f).replace(".csv", "")
        df["model"] = model_name   # 新增来源列
        dfs.append(df)

    merged = pd.concat(dfs, axis=0, ignore_index=True)
    merged.to_csv(merged_out, index=False)
    return merged_out


# example:
# out = run_species_metadata_pipeline("human", ["title", "summary"])
# print("Saved to", out)

In [2]:
# human title sum
out_1 = run_species_metadata_pipeline("human", ["title", "summary"])
print("Saved to", out_1)

Models: 100%|██████████| 1166/1166 [00:03<00:00, 305.87it/s]


Saved to results/human_title_summary_preds.csv


In [3]:
# human title sum
out_2 = run_species_metadata_pipeline("human", ["title", "summary", "design"])
print("Saved to", out_2)

Models: 100%|██████████| 1166/1166 [00:04<00:00, 277.98it/s]


Saved to results/human_title_summary_design_preds.csv


In [4]:
# mouse title sum design
out_3 = run_species_metadata_pipeline("mouse", ["title", "summary"])
print("Saved to", out_3)

Models: 100%|██████████| 1166/1166 [00:04<00:00, 272.15it/s]


Saved to results/mouse_title_summary_preds.csv


In [5]:
# mouse title sum design
out_4 = run_species_metadata_pipeline("mouse", ["title", "summary", "design"])
print("Saved to", out_4)

Models: 100%|██████████| 1166/1166 [6:09:02<00:00, 18.99s/it] 


Saved to results/mouse_title_summary_design_preds.csv


# History

## human

### title + summary

In [1]:
%%bash
python src/preprocess.py \
    -input demo/human_title_summary.tsv \
    -out demo/human_title_summary_processed.tsv

Models:   7%|▋         | 82/1166 [00:18<00:01, 816.25it/s]

In [2]:
%%bash
python src/embedding_lookup_table.py \
    -input demo/human_title_summary_processed.tsv \
    -out demo/human_title_summary_embedding.npz

get unique words
there are 24081 unique words in the given descriptions
generate embedding by cpu


generate embeddings using cpu: 100%|█████| 24081/24081 [09:28<00:00, 42.35it/s]


saving output...


In [14]:
# %%bash
# python src/predict.py \
#     -input demo/human_title_summary_processed.tsv \
#     -id demo/human_ID.tsv \
#     -input_embed demo/human_title_summary_embedding.npz \
#     -train_embed data/disease_desc_embedding.npz \
#     -model bins/MONDO_0000167__model.pkl \
#     -out results/human_title_summary/

In [17]:
# import glob
# import subprocess
# import os
# from tqdm import tqdm

# os.makedirs("results/human_title_summary", exist_ok=True)

# models = glob.glob("bins/MONDO_*__model.pkl")

# for model in tqdm(models, desc="Running models"):
#     subprocess.run([
#         "python", "src/predict.py",
#         "-input", "demo/human_title_summary_processed.tsv",
#         "-id", "demo/human_ID.tsv",
#         "-input_embed", "demo/human_title_summary_embedding.npz",
#         "-train_embed", "data/disease_desc_embedding.npz",
#         "-model", model,
#         "-out", "results/human_title_summary/"
#     ], check=True)

Running models:   0%|          | 0/1166 [00:00<?, ?it/s]/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 1.13 s to load data
predicting labels
took 15.19 s to predict
retrieving predictive words
took 0.65 s to retrieve predictive words
saving output
took 0.28 min to load, predict, retrieve predictive words and save MONDO_0000167 for 3395 instances


Running models:   0%|          | 1/1166 [00:18<6:08:48, 18.99s/it]/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 1.14 s to load data
predicting labels
took 14.84 s to predict
retrieving predictive words
took 0.40 s to retrieve predictive words
saving output
took 0.27 min to load, predict, retrieve predictive words and save MONDO_0000248 for 3395 instances


Running models:   0%|          | 2/1166 [00:37<6:03:15, 18.72s/it]/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


...

loading data
took 1.12 s to load data
predicting labels
took 14.62 s to predict
retrieving predictive words
took 0.42 s to retrieve predictive words
saving output
took 0.27 min to load, predict, retrieve predictive words and save MONDO_0001347 for 3395 instances


Running models:   7%|▋         | 84/1166 [26:05<5:34:15, 18.54s/it]/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 1.15 s to load data
predicting labels
took 14.71 s to predict
retrieving predictive words
took 0.48 s to retrieve predictive words
saving output
took 0.27 min to load, predict, retrieve predictive words and save MONDO_0001358 for 3395 instances


Running models:   7%|▋         | 85/1166 [26:23<5:31:59, 18.43s/it]/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Running models:   7%|▋         | 85/1166 [26:28<5:36:38, 18.69s/it]


KeyboardInterrupt: 

In [ ]:
import glob, os, subprocess, datetime
from tqdm import tqdm

models = sorted(glob.glob("bins/MONDO_*__model.pkl"))
out_dir = "results/human_title_summary"
log_path = "results/human_title_summary/predict_all.log"

os.makedirs(out_dir, exist_ok=True)
os.makedirs(os.path.dirname(log_path), exist_ok=True)

for model in tqdm(models, desc="Models"):
    mondo = os.path.basename(model).split("__model.pkl")[0]
    done = glob.glob(os.path.join(out_dir, f"{mondo}*.csv"))
    if done:
        continue

    with open(log_path, "a", encoding="utf-8") as log:
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log.write(f"\n[{ts}] START {mondo}\n")
        log.flush()

        try:
            subprocess.run([
                "python", "src/predict.py",
                "-input", "demo/human_title_summary_processed.tsv",
                "-id", "demo/human_ID.tsv",
                "-input_embed", "demo/human_title_summary_embedding.npz",
                "-train_embed", "data/disease_desc_embedding.npz",
                "-model", model,
                "-out", out_dir + "/"
            ], stdout=log, stderr=log, check=True)

            ts2 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"[{ts2}] DONE  {mondo}\n")
            log.flush()

        except subprocess.CalledProcessError as e:
            ts3 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"[{ts3}] FAIL  {mondo} (exit={e.returncode})\n")
            log.flush()
            raise


In [ ]:
import pandas as pd

in_dir = "results/human_title_summary"
out_file = "results/human_title_summary_preds.csv"

files = sorted(glob.glob(os.path.join(in_dir, "*.csv")))

dfs = []
for f in files:
    df = pd.read_csv(f)

    model_name = os.path.basename(f).replace(".csv", "")
    df.columns = [model_name]   # 用文件名作为列名

    dfs.append(df)

merged = pd.concat(dfs, axis=1)
merged.to_csv(out_file, index=False)

print(f"Saved to {out_file}")

### title + summary + design

In [ ]:
import pandas as pd

in_dir = "results/human_title_summary_design"
out_file = "results/human_title_summary_design_preds.csv"

files = sorted(glob.glob(os.path.join(in_dir, "*.csv")))

dfs = []
for f in files:
    df = pd.read_csv(f)

    model_name = os.path.basename(f).replace(".csv", "")
    df.columns = [model_name]   # 用文件名作为列名

    dfs.append(df)

merged = pd.concat(dfs, axis=1)
merged.to_csv(out_file, index=False)

print(f"Saved to {out_file}")

## mouse
### title + summary

In [ ]:
%%bash
python src/preprocess.py \
    -input demo/human_title_summary.tsv \
    -out demo/human_title_summary_processed.tsv

In [ ]:
%%bash
python src/embedding_lookup_table.py \
    -input demo/human_title_summary_processed.tsv \
    -out demo/human_title_summary_embedding.npz

In [ ]:
import glob, os, subprocess, datetime
from tqdm import tqdm

models = sorted(glob.glob("bins/MONDO_*__model.pkl"))
out_dir = "results/human_title_summary"
log_path = "results/predict_all.log"

os.makedirs(out_dir, exist_ok=True)
os.makedirs(os.path.dirname(log_path), exist_ok=True)

for model in tqdm(models, desc="Models"):
    mondo = os.path.basename(model).split("__model.pkl")[0]
    done = glob.glob(os.path.join(out_dir, f"{mondo}*.csv"))
    if done:
        continue

    with open(log_path, "a", encoding="utf-8") as log:
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log.write(f"\n[{ts}] START {mondo}\n")
        log.flush()

        try:
            subprocess.run([
                "python", "src/predict.py",
                "-input", "demo/human_title_summary_processed.tsv",
                "-id", "demo/human_ID.tsv",
                "-input_embed", "demo/human_title_summary_embedding.npz",
                "-train_embed", "data/disease_desc_embedding.npz",
                "-model", model,
                "-out", out_dir + "/"
            ], stdout=log, stderr=log, check=True)

            ts2 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"[{ts2}] DONE  {mondo}\n")
            log.flush()

        except subprocess.CalledProcessError as e:
            ts3 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"[{ts3}] FAIL  {mondo} (exit={e.returncode})\n")
            log.flush()
            raise


In [ ]:
import pandas as pd

in_dir = "results/mouse_title_summary"
out_file = "results/mouse_title_summary_preds.csv"

files = sorted(glob.glob(os.path.join(in_dir, "*.csv")))

dfs = []
for f in files:
    df = pd.read_csv(f)

    model_name = os.path.basename(f).replace(".csv", "")
    df.columns = [model_name]   # 用文件名作为列名

    dfs.append(df)

merged = pd.concat(dfs, axis=1)
merged.to_csv(out_file, index=False)

print(f"Saved to {out_file}")

### title + summary + design

In [ ]:
%%bash
python src/preprocess.py \
    -input demo/human_title_summary.tsv \
    -out demo/human_title_summary_processed.tsv

In [ ]:
%%bash
python src/embedding_lookup_table.py \
    -input demo/human_title_summary_processed.tsv \
    -out demo/human_title_summary_embedding.npz

In [ ]:
import glob, os, subprocess, datetime
from tqdm import tqdm

models = sorted(glob.glob("bins/MONDO_*__model.pkl"))
out_dir = "results/human_title_summary"
log_path = "results/predict_all.log"

os.makedirs(out_dir, exist_ok=True)
os.makedirs(os.path.dirname(log_path), exist_ok=True)

for model in tqdm(models, desc="Models"):
    mondo = os.path.basename(model).split("__model.pkl")[0]
    done = glob.glob(os.path.join(out_dir, f"{mondo}*.csv"))
    if done:
        continue

    with open(log_path, "a", encoding="utf-8") as log:
        ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log.write(f"\n[{ts}] START {mondo}\n")
        log.flush()

        try:
            subprocess.run([
                "python", "src/predict.py",
                "-input", "demo/human_title_summary_processed.tsv",
                "-id", "demo/human_ID.tsv",
                "-input_embed", "demo/human_title_summary_embedding.npz",
                "-train_embed", "data/disease_desc_embedding.npz",
                "-model", model,
                "-out", out_dir + "/"
            ], stdout=log, stderr=log, check=True)

            ts2 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"[{ts2}] DONE  {mondo}\n")
            log.flush()

        except subprocess.CalledProcessError as e:
            ts3 = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log.write(f"[{ts3}] FAIL  {mondo} (exit={e.returncode})\n")
            log.flush()
            raise


In [ ]:
import pandas as pd

in_dir = "results/mouse_title_summary_design"
out_file = "results/mouse_title_summary_design_preds.csv"

files = sorted(glob.glob(os.path.join(in_dir, "*.csv")))

dfs = []
for f in files:
    df = pd.read_csv(f)

    model_name = os.path.basename(f).replace(".csv", "")
    df.columns = [model_name]   # 用文件名作为列名

    dfs.append(df)

merged = pd.concat(dfs, axis=1)
merged.to_csv(out_file, index=False)

print(f"Saved to {out_file}")